In [3]:
# Import libraries for data processing and file handling
import pandas as pd
import json
import re
from pathlib import Path

# Define input and output locations
RAW_DIR = Path(".")
OUTPUT_DIR = Path("processed")

BLS_FILE = RAW_DIR / "bls_occupational_projections_2025_2035 1.2.csv"
SOC_FILE = RAW_DIR / "soc_2018_definitions.xlsx"

# Create the output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
import duckdb

con = duckdb.connect()

con.execute("""
    CREATE OR REPLACE VIEW employment AS
    SELECT *
    FROM read_csv(
        'bls_occupational_projections_2025_2035 1.2.csv',
        delim=',',
        header=True,
        skip=1,
        auto_detect=True
    )
""")

In [9]:
df = con.execute(
    """
    select *
    from employment e 
    """).df()

In [11]:
# Copy the existing cleaned DataFrame
bls = df.copy()

# Normalize column names to lowercase snake_case
bls.columns = (
    bls.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Inspect the columns
print(bls.columns.tolist())
print("Number of rows:", len(bls))

['2025_national_employment_matrix_title', '2025_national_employment_matrix_code', 'occupation_type', 'employment_2025', 'employment_2035', 'employment_distribution_percent_2025', 'employment_distribution_percent_2035', 'employment_change_numeric_2025_35', 'employment_change_percent_2025_35', 'percent_self_employed_2025', 'occupational_openings_2025_35_annual_average', 'median_annual_wage_dollars_2025', 'typical_education_needed_for_entry', 'work_experience_in_a_related_occupation', 'typical_on_the_job_training_needed_to_attain_competency_in_the_occupation', 'related_occupational_outlook_handbook_ooh_content', 'related_occupational_outlook_handbook_ooh_content_1']
Number of rows: 1117


In [13]:
# Inspect all available sheets in the official SOC workbook
xls = pd.ExcelFile(SOC_FILE)

print("Sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    preview = pd.read_excel(
        SOC_FILE,
        sheet_name=sheet,
        header=None,
        nrows=8
    )

    print(f"\n--- {sheet} ---")
    display(preview)

Sheets: ['2018 SOC']

--- 2018 SOC ---


,0,1,2,3
0,U.S. Bureau of Labor Statistics,NaN,NaN,NaN
1,On behalf of the Office of Management and Budg...,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,November 2017 (for reference year January 2018...,NaN,NaN,NaN
4,***This is the final structure and detailed de...,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,SOC Group,SOC Code,SOC Title,SOC Definition


In [19]:
# Identify the occupation code column
CODE_COL = "2025_national_employment_matrix_code"
TITLE_COL = "2025_national_employment_matrix_title"

# Validate required columns
assert CODE_COL in bls.columns
assert TITLE_COL in bls.columns

# Create convenient standardized columns
bls["occupation_code"] = bls[CODE_COL].astype(str).str.strip()
bls["occupation_title"] = bls[TITLE_COL].astype(str).str.strip()

# Determine the SOC hierarchy level from the code
def get_soc_level(code):
    if not re.fullmatch(r"\d{2}-\d{4}", code):
        return "unknown"
    if code.endswith("0000"):
        return "major"
    if code.endswith("00"):
        return "minor"
    if code.endswith("0"):
        return "broad"
    return "detailed"

bls["soc_level"] = bls["occupation_code"].apply(get_soc_level)

# Inspect the resulting distribution
print(bls["soc_level"].value_counts())

soc_level
detailed    824
broad       170
minor        95
major        23
unknown       5
Name: count, dtype: int64


In [17]:
# Display all column names in the DataFrame
print(bls.columns.tolist())

['2025_national_employment_matrix_title', '2025_national_employment_matrix_code', 'occupation_type', 'employment_2025', 'employment_2035', 'employment_distribution_percent_2025', 'employment_distribution_percent_2035', 'employment_change_numeric_2025_35', 'employment_change_percent_2025_35', 'percent_self_employed_2025', 'occupational_openings_2025_35_annual_average', 'median_annual_wage_dollars_2025', 'typical_education_needed_for_entry', 'work_experience_in_a_related_occupation', 'typical_on_the_job_training_needed_to_attain_competency_in_the_occupation', 'related_occupational_outlook_handbook_ooh_content', 'related_occupational_outlook_handbook_ooh_content_1']


In [21]:
# Return the expected parent code based on SOC structure
def get_expected_parent(code, level):
    if level == "major":
        return None
    if level == "minor":
        return code[:2] + "-0000"
    if level == "broad":
        return code[:5] + "00"
    if level == "detailed":
        return code[:6] + "0"
    return None

bls["expected_parent_code"] = bls.apply(
    lambda row: get_expected_parent(
        row["occupation_code"],
        row["soc_level"]
    ),
    axis=1
)

# Check which expected parents actually exist in our dataset
available_codes = set(bls["occupation_code"])

bls["parent_code"] = bls["expected_parent_code"].apply(
    lambda code: code if code in available_codes else None
)

# Flag missing parent nodes for further inspection
bls["missing_parent"] = (
    bls["expected_parent_code"].notna()
    & bls["parent_code"].isna()
)

display(
    bls.loc[
        bls["missing_parent"],
        ["occupation_code", "occupation_title",
         "soc_level", "expected_parent_code"]
    ]
)

,occupation_code,occupation_title,soc_level,expected_parent_code
3,11-1011,Chief executives,detailed,11-1010
4,11-1021,General and operations managers,detailed,11-1020
6,11-2011,Advertising and promotions managers,detailed,11-2010
17,11-3021,Computer and information systems managers,detailed,11-3020
18,11-3031,Financial managers,detailed,11-3030
...,...,...,...,...
1097,53-7041,Hoist and winch operators,detailed,53-7040
1098,53-7051,Industrial truck and tractor operators,detailed,53-7050
1109,53-7081,Refuse and recyclable material collectors,detailed,53-7080
1110,53-7121,"Tank car, truck, and ship loaders",detailed,53-7120


In [23]:
# Extract SOC codes and titles from the official workbook
soc_records = []
code_pattern = re.compile(r"^\d{2}-\d{4}$")

for sheet in xls.sheet_names:
    sheet_df = pd.read_excel(
        SOC_FILE,
        sheet_name=sheet,
        header=None,
        dtype=str
    )

    for row in sheet_df.itertuples(index=False, name=None):
        values = [
            str(v).strip() if pd.notna(v) else ""
            for v in row
        ]

        for i, value in enumerate(values):
            if code_pattern.fullmatch(value):
                title = values[i + 1] if i + 1 < len(values) else ""

                if title:
                    soc_records.append({
                        "occupation_code": value,
                        "soc_official_title": title
                    })
                break

soc = pd.DataFrame(
    soc_records,
    columns=["occupation_code", "soc_official_title"]
).drop_duplicates("occupation_code")

# Match the official SOC records to BLS occupations
bls = bls.merge(
    soc,
    on="occupation_code",
    how="left",
    validate="many_to_one"
)

bls["soc_match"] = bls["soc_official_title"].notna()

print("SOC records found:", len(soc))
print("Matched BLS rows:", bls["soc_match"].sum())
print("Unmatched BLS rows:", (~bls["soc_match"]).sum())

SOC records found: 1447
Matched BLS rows: 1106
Unmatched BLS rows: 11


In [27]:

# Display unmatched occupations that have valid SOC-style codes
unmatched_detailed = bls.loc[
    (~bls["soc_match"])
    & (bls["soc_level"] == "detailed"),
    ["occupation_code", "occupation_title", "occupation_type"]
]

display(unmatched_detailed)

,occupation_code,occupation_title,occupation_type
236,21-1018,"Substance abuse, behavioral disorder, and ment...",Line item
329,25-2052,"Special education teachers, kindergarten and e...",Line item
351,25-9045,"Teaching assistants, except postsecondary",Line item
912,51-2028,"Electrical, electronic, and electromechanical ...",Line item
1048,53-1047,First-line supervisors of transportation and m...,Line item


In [29]:
# Find the closest available parent for unmatched detailed occupations
def find_parent(code, available_codes):
    if not re.fullmatch(r"\d{2}-\d{4}", str(code)):
        return None

    candidates = [
        code[:-1] + "0",    # Broad occupation
        code[:5] + "00",    # Minor group
        code[:2] + "-0000"  # Major group
    ]

    for parent in candidates:
        if parent != code and parent in available_codes:
            return parent

    return None


# Find the five unmatched detailed occupations
mask = (
    (~bls["soc_match"])
    & (bls["soc_level"] == "detailed")
)

available_codes = set(bls["occupation_code"])

# Suggest a parent for each occupation
bls.loc[mask, "parent_code"] = bls.loc[
    mask, "occupation_code"
].apply(lambda code: find_parent(code, available_codes))

# Display the suggested assignments
display(
    bls.loc[
        mask,
        ["occupation_code", "occupation_title", "parent_code"]
    ]
)

,occupation_code,occupation_title,parent_code
236,21-1018,"Substance abuse, behavioral disorder, and ment...",21-1010
329,25-2052,"Special education teachers, kindergarten and e...",25-2050
351,25-9045,"Teaching assistants, except postsecondary",25-9040
912,51-2028,"Electrical, electronic, and electromechanical ...",51-2020
1048,53-1047,First-line supervisors of transportation and m...,53-1000


In [31]:
# Create a lookup table for occupation titles
title_lookup = bls.set_index(
    "occupation_code"
)["occupation_title"].to_dict()

# Display unmatched occupations alongside their suggested parents
review = bls.loc[
    mask,
    ["occupation_code", "occupation_title", "parent_code"]
].copy()

review["parent_title"] = review["parent_code"].map(title_lookup)

display(review)

,occupation_code,occupation_title,parent_code,parent_title
236,21-1018,"Substance abuse, behavioral disorder, and ment...",21-1010,Counselors
329,25-2052,"Special education teachers, kindergarten and e...",25-2050,Special education teachers
351,25-9045,"Teaching assistants, except postsecondary",25-9040,Teaching assistants
912,51-2028,"Electrical, electronic, and electromechanical ...",51-2020,"Electrical, electronics, and electromechanical..."
1048,53-1047,First-line supervisors of transportation and m...,53-1000,Supervisors of transportation and material mov...


In [33]:
# Mark the hierarchy source for traceability
bls["hierarchy_source"] = "soc_2018"

bls.loc[
    ~bls["soc_match"],
    "hierarchy_source"
] = "unresolved"

# Mark reviewed manual assignments
bls.loc[
    mask & bls["parent_code"].notna(),
    "hierarchy_source"
] = "manual_verified"

# Update the missing-parent indicator
bls["missing_parent"] = (
    bls["soc_level"].ne("major")
    & bls["parent_code"].isna()
)

# Verify that all five occupations have parents
assert bls.loc[mask, "parent_code"].notna().all()

In [41]:
# Find the closest available ancestor using SOC code structure
def find_closest_parent(code, available_codes):
    if not isinstance(code, str):
        return None

    if not re.fullmatch(r"\d{2}-\d{4}", code):
        return None

    if code == "00-0000" or code.endswith("0000"):
        return None

    candidates = [
        code[:-1] + "0",       # Broad occupation
        code[:5] + "00",       # Minor group
        code[:2] + "-0000"     # Major group
    ]

    for parent in dict.fromkeys(candidates):
        if parent != code and parent in available_codes:
            return parent

    return None


# Recalculate parent codes for all occupations
available_codes = set(bls["occupation_code"])

bls["parent_code"] = bls["occupation_code"].apply(
    lambda code: find_closest_parent(code, available_codes)
)

# Recalculate missing-parent indicators
bls["missing_parent"] = (
    bls["soc_level"].isin(["minor", "broad", "detailed"])
    & bls["parent_code"].isna()
)

In [43]:
# Count occupations still missing a parent
missing_parents = bls[bls["missing_parent"]]

print("Missing parents:", len(missing_parents))

# Show missing parents by hierarchy level
display(
    missing_parents.groupby("soc_level")
    .size()
    .reset_index(name="count")
)

# Verify parent assignments for sample occupations
display(
    bls.loc[
        bls["occupation_code"].isin([
            "11-1011",
            "11-1021",
            "15-1250",
            "15-1252"
        ]),
        ["occupation_code", "occupation_title",
         "soc_level", "parent_code"]
    ]
)

Missing parents: 0


,soc_level,count


,occupation_code,occupation_title,soc_level,parent_code
3,11-1011,Chief executives,detailed,11-1000
4,11-1021,General and operations managers,detailed,11-1000
104,15-1250,"Software and web developers, programmers, and ...",broad,15-1200
106,15-1252,Software developers,detailed,15-1250


In [45]:
# Validate that all hierarchy paths reach a major group
parent_lookup = (
    bls.dropna(subset=["occupation_code"])
    .drop_duplicates("occupation_code")
    .set_index("occupation_code")["parent_code"]
    .to_dict()
)

def validate_path(code):
    visited = set()
    current = code

    while pd.notna(current):
        if current in visited:
            return "cycle"

        visited.add(current)

        if current not in parent_lookup:
            return "missing_node"

        parent = parent_lookup[current]

        if pd.isna(parent):
            return "valid" if current.endswith("-0000") else "orphan"

        current = parent

    return "orphan"


# Check every occupation with a valid SOC-style code
valid_codes = bls.loc[
    bls["occupation_code"].str.fullmatch(r"\d{2}-\d{4}", na=False),
    "occupation_code"
].unique()

results = pd.Series(
    [validate_path(code) for code in valid_codes]
)

print(results.value_counts())

valid    1112
Name: count, dtype: int64


In [47]:
# Define the output file path
CSV_OUTPUT = OUTPUT_DIR / "bls_occupational_projections_clean.csv"

# Save all original and added columns to a new CSV file
bls.to_csv(
    CSV_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

# Verify the saved file
saved_df = pd.read_csv(CSV_OUTPUT, dtype={"occupation_code": str})

print("File saved:", CSV_OUTPUT)
print("Rows:", len(saved_df))
print("Columns:", len(saved_df.columns))
print("Hierarchy columns:",
      [col for col in ["soc_level", "parent_code", "soc_match"]
       if col in saved_df.columns])

File saved: processed\bls_occupational_projections_clean.csv
Rows: 1117
Columns: 26
Hierarchy columns: ['soc_level', 'parent_code', 'soc_match']


In [49]:
# Keep records with valid occupation codes
tree_df = bls[
    bls["occupation_code"].str.fullmatch(
        r"\d{2}-\d{4}", na=False
    )
].copy()

# Exclude the overall employment total
tree_df = tree_df[
    tree_df["occupation_code"] != "00-0000"
]

# Create one node for each occupation
nodes = {}

for _, row in tree_df.iterrows():
    code = row["occupation_code"]

    nodes[code] = {
        "code": code,
        "name": row["occupation_title"],
        "level": row["soc_level"],
        "occupation_type": row["occupation_type"],
        "children": []
    }

print("Nodes created:", len(nodes))

Nodes created: 1111


In [51]:
# Build the occupation hierarchy
roots = []

for _, row in tree_df.iterrows():
    code = row["occupation_code"]
    parent = row["parent_code"]

    if pd.notna(parent) and parent in nodes:
        nodes[parent]["children"].append(nodes[code])
    else:
        roots.append(nodes[code])

print("Major groups:", len(roots))

Major groups: 22


In [53]:
# Count nodes recursively
def count_nodes(tree):
    return sum(
        1 + count_nodes(node["children"])
        for node in tree
    )

total_nodes = count_nodes(roots)

print("Expected:", len(tree_df))
print("Actual:", total_nodes)

assert total_nodes == len(tree_df)

# Display root categories
for root in roots:
    print(root["code"], "-", root["name"])

Expected: 1111
Actual: 1111
11-0000 - Management occupations
13-0000 - Business and financial operations occupations
15-0000 - Computer and mathematical occupations
17-0000 - Architecture and engineering occupations
19-0000 - Life, physical, and social science occupations
21-0000 - Community and social service occupations
23-0000 - Legal occupations
25-0000 - Educational instruction and library occupations
27-0000 - Arts, design, entertainment, sports, and media occupations
29-0000 - Healthcare practitioners and technical occupations
31-0000 - Healthcare support occupations
33-0000 - Protective service occupations
35-0000 - Food preparation and serving related occupations
37-0000 - Building and grounds cleaning and maintenance occupations
39-0000 - Personal care and service occupations
41-0000 - Sales and related occupations
43-0000 - Office and administrative support occupations
45-0000 - Farming, fishing, and forestry occupations
47-0000 - Construction and extraction occupations
49-0

In [55]:
# Save the hierarchy as a JSON file
JSON_OUTPUT = OUTPUT_DIR / "bls_occupation_hierarchy.json"

with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(roots, f, ensure_ascii=False, indent=2)

print("JSON saved:", JSON_OUTPUT)

JSON saved: processed\bls_occupation_hierarchy.json
